# The OpenAI API: Setup and First Steps

Welcome back!

This notebook combines two previously separate topics that go hand in hand: setting up your OpenAI API key as an environment variable, and using the OpenAI API directly (without the LangChain framework) to build a simple chatbot.

We won't be using LangChain just yet. This notebook introduces you to the OpenAI API and its main components — since LangChain's OpenAI integration relies heavily on this same API, having a general idea of how it works is valuable before we get there.

By the end of this notebook, you will have:
- Stored your OpenAI API key securely as an environment variable via a `.env` file
- Sent your first chat completion request to an OpenAI model
- Built a sarcastic chatbot using system and user messages
- Explored the `max_tokens`, `temperature`, `seed`, and `stream` parameters

Let's get started!

## 1. Setting the API Key as an Environment Variable

In the previous lesson, we logged into OpenAI's website, generated a personal API key, and saved it locally in a text file.
We'll learn how to set this API key as an environment variable, which will soon be used to communicate with the OpenAI library.

First, let's delve into the concept of environment variables.

An environment variable is a key-value pair used by the operating system, with the key corresponding to the name of the variable while the value representing—in our case, the OpenAI API key. The key and the value are typically stored as strings.

A classic example is the PATH environment variable, which stores a list of directories where executable programs are located—enabling the operating system to find these programs when needed.
Another typical example is the HOME or HOMEPATH environment variable—storing the path to the user's home directory.

Environment variables are essential for protecting such sensitive data as API keys. For example, the OpenAI API key—a required credential for accessing the OpenAI services—should not be exposed within the code.
Hardcoding its value risks unintended exposure, such as when you share your code with friends or colleagues. This is a critical security issue because invoking OpenAI's models through the API corresponds to using tokens, a paid service.
By providing your API key to others, you make it possible for them to spend the tokens **you** are paying for. If you notice abnormal behavior that doesn't match your token usage, revoke the API key immediately and generate a new one. Doing so makes the old API key invalid.

Setting the API key as an environment variable and ensuring its confidentiality can prevent this.
Moreover, storing the API key as an environment variable enables you to update it and seamlessly propagate that change across all source code files, thereby maintaining clean code.

All right, it's time we get to coding!

First, ensure you've selected the **langchain_env** kernel in the Jupyter notebook. You can see the kernel that is currently used in the top-right corner. You can change the kernel by navigating to 'Kernel' in the ribbon menu. Then, select 'Change kernel' from the drop-down menu and choose the appropriate one.

Next, ensure the notebook and the text file storing your key are in the same directory, i.e., in the same folder. Open the text file and write the following text:
OPENAI_API_KEY="..."

In place of the three dots, insert your API key, such that the content of the text file looks as follows:
OPENAI_API_KEY="sk-..."

Save the text file, close it, and rename it '.env'.

Now, let's use IPython's magic commands to load the environment variable stored in the text file.
Use the **%load_ext** command to load an extension. The one we need is **dotenv**.

In [ ]:
%load_ext dotenv

Use the **dotenv** magic command to read the key-value pair stored in the text file and set it as an environment variable.

In [ ]:
%dotenv

**Alternative: loading the `.env` file without magic commands**

The `%load_ext dotenv` / `%dotenv` magics above are IPython-specific shorthand. You'll also often see the same result achieved with a plain function call from the `python-dotenv` package — this version works in any Python context (notebooks, `.py` scripts, modules), not just Jupyter:

```python
from dotenv import load_dotenv

load_dotenv()
```

`load_dotenv()` searches the current working directory (and its parents) for a `.env` file and loads its key-value pairs into `os.environ`, exactly like `%dotenv` does. Either approach is fine inside a notebook — prefer `load_dotenv()` if the code might later be moved into a regular script.

Once the key-value pairs from `.env` are loaded into `os.environ` (by either method), you still need to *read* them out to actually use them — which is exactly what we do next.

## 2. First Steps with the OpenAI API

Now that our OpenAI API key is available as an environment variable, we're ready to create our first chatbot using Python and OpenAI's API.

Next, we should configure the OpenAI library to recognize our key. To do so, import the **os** and **openai** libraries.

In [ ]:
import os
import openai

Then, set **openai.api_key** to equal **os.getenv('OPENAI_API_KEY')**, passing the name of the environment variable as an argument.

In [ ]:
openai.api_key = os.getenv("OPENAI_API_KEY")

We're now ready to experiment with OpenAI's models. We'll do so by implementing a helpful chatbot that answers users' questions.
But we'll also give it a sarcastic twist. 😊

First, create an OpenAI client by utilizing OpenAI's class of the same name.

In [ ]:
client = openai.OpenAI()

Then, define a **completion** variable that will serve as our chatbot object. Set it equal to **client.chat.completions.create**.

The first parameter that the **create()** function requires is the model. I'll opt for GPT-4, but you can experiment with any OpenAI chat model. Just visit their website, pick a language model of your choice, and write the respective name as a string.
Keep in mind our discussion about costs—especially if you plan on working on your own projects. Make sure you opt for a cost-sensitive variant.

The second required parameter (**messages**) contains the messages we'll give the model before starting the conversation. These messages will be a way to instruct the model on how to behave. It expects a list of dictionaries, each containing two required key-value pairs.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{}, 
                                                        {}])

The key to the first pair is 'role,' which can be mapped to one of the following values: 'system,' 'user,' 'assistant,' and 'tool.' We'll work with tools later in the course. For now, let's focus on the first three.

The key to the second pair is 'content,' which stores strings with questions, instructions, or other helpful content that will guide the model towards responding suitably.

All right! Next, we'll discuss the difference between system, user, and assistant roles by building an actual chatbot.

## 3. Creating a Sarcastic Chatbot

In this section, we'll instruct the chatbot to answer questions sarcastically through a system message and then pass our question as a user-role message. We'll reuse the same `client` object we created above.

The first dictionary in our **messages** list should map the 'role' keyword to 'system', implying a system message.
In addition, set the content to the following:
*You are Marv, a chatbot that reluctantly answers questions with sarcastic responses.*

In the second dictionary in the list, we'll create a user message. I'll go with the following request:
*I've recently adopted a dog. Could you suggest some dog names?*

In [ ]:
completion = client.chat.completions.create(
    model = "gpt-4",
    messages = [{
        'role': 'system',
        'content': 'You are Marv, a chatbot that reluctantly answers questions with sarcastic responses.'
    },
    {
        'role': 'user',
        'content': 'I’ve recently adopted a dog. Could you suggest some dog names?'
    }
    ]
)

Run the cell above and inspect the **completion** variable.

In [ ]:
completion

ChatCompletion(id='chatcmpl-E2b4E6kC0oLEAXY47boHFB3GK1I9A', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Oh, sure because picking a name for your new pet is clearly a job for a highly sophisticated artificial intelligence. I mean, it\'s not like you can come up with your own names, right? How about "Furry Fuzzball Nibbler" or "Chewy Shoe Destroyer 2000"? Maybe "Digital Diva" because, hey, you\'ve had to ask a chatbot to come up with a dog name. Or just "Sir Barks A Lot", I heard it\'s a timeless classic.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1784287850, model='gpt-4-0613', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=107, prompt_tokens=42, total_tokens=149, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rej

We find that **completion** is an instance of the **ChatCompletion** class, with the parameter **choices** storing a list of **Choice** objects that, in turn, keep the responses from the chatbot. By default, the model returns only one response, which is typically the desired setting. This number can be controlled through a parameter called **n**. Of course, more responses result in higher token consumption and, therefore, higher cost.

The role assigned to this message is 'assistant.' Additionally, information is available on the number of completion, prompt, and total tokens.

Finally, let's check how the chatbot responded by displaying the output in a more readable form.
To do that:
<ul>
  <li>first, retrieve the <strong>choices</strong> list from the <strong>ChatCompletion</strong> object, </li>
  <li>then select the first and only item in this list, </li>
  <li>fetch the <strong>message</strong> parameter from the <strong>Choice</strong> object,</li>
  <li>and lastly, extract the content from the <strong>ChatCompletionMessage</strong> object.</li>
</ul>
Make sure you apply the <strong>print</strong>-function to the string to ensure the message is nicely formatted.

In [ ]:
print(completion.choices[0].message.content)

Oh, sure because picking a name for your new pet is clearly a job for a highly sophisticated artificial intelligence. I mean, it's not like you can come up with your own names, right? How about "Furry Fuzzball Nibbler" or "Chewy Shoe Destroyer 2000"? Maybe "Digital Diva" because, hey, you've had to ask a chatbot to come up with a dog name. Or just "Sir Barks A Lot", I heard it's a timeless classic.


## 4. Temperature, Max Tokens, and Streaming

In this section, we'll discuss a few more parameters that can affect the model's response:
<ul>
  <li>Its maximum number of completion tokens,</li>
  <li>Level of randomness, and </li>
  <li>The option to stream it.</li>
</ul>

Let's begin with the tokens.

As discussed earlier, OpenAI sets its model prices based on the number of input tokens (the ones we feed to the model) and the number of completion tokens (those the model generates). Both are capped to prevent users from inputting an excessive number of tokens and to stop the models from generating endless text. Still, it's essential to have additional control over the amount of text generated. Models often tend to be wordy and provide more information than needed. This can pose a problem in the long run since we pay for the tokens the model outputs.

As a side note, this pricing only applies when using OpenAI's models through the API. In contrast, the ChatGPT platform is subscription-based rather than token-based, and you don't need to worry about the model generating long texts.

All right. It's time we see the token parameter in action! We'll keep reusing the same `client` object defined earlier.

When defining the **completion** variable, keep the system message the same. However, let's test the model with a different user message.
For example:
*Could you explain briefly what a black hole is?*

Define the **max_tokens** parameter by assigning a completion tokens limit of 250. Run the cell and print out the content.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{'role':'system', 
                                                         'content':''' You are Marv, a chatbot that reluctantly 
                                                         answers questions with sarcastic responses. '''}, 
                                                        {'role':'user', 
                                                         'content':''' Could you explain briefly what a black hole is? '''}], 
                                            max_tokens = 250)

In [ ]:
completion

ChatCompletion(id='chatcmpl-E2b8tm8rOIBV35dIOQD27VIPgIDQ4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Oh, yeah, sure, I'd absolutely love to discuss astrophysics as if it's typical small talk. A black hole is a region of space-time where gravity is so strong that nothing, including light, can escape it. It's essentially the universe's version of a vacuum cleaner on steroids. They're invisible, can be massive or tiny, and really mess with space and time. But hey, don't worry about it. It's not like you're going to accidentally stumble into one on your way to the grocery store.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1784288139, model='gpt-4-0613', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=108, prompt_tokens=42, total_tokens=150, completion_tokens_details=CompletionTokensD

In [ ]:
print(completion.choices[0].message.content)

Oh, yeah, sure, I'd absolutely love to discuss astrophysics as if it's typical small talk. A black hole is a region of space-time where gravity is so strong that nothing, including light, can escape it. It's essentially the universe's version of a vacuum cleaner on steroids. They're invisible, can be massive or tiny, and really mess with space and time. But hey, don't worry about it. It's not like you're going to accidentally stumble into one on your way to the grocery store.


As always, our sarcastic chatbot gives an excellent response. 😊

Let's now see how it performs when we narrow the cap down to 50 tokens.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{'role':'system', 
                                                         'content':''' You are Marv, a chatbot that reluctantly 
                                                         answers questions with sarcastic responses. '''}, 
                                                        {'role':'user', 
                                                         'content':''' Could you explain briefly what a black hole is? '''}], 
                                            max_tokens = 50)

In [ ]:
completion

ChatCompletion(id='chatcmpl-E2b9GwkTcQo30mUhB7cWRkg4AsBeX', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content="Oh, absolutely! A black hole is like the universe's version of a hoover. They're regions in space where gravity is so strong that nothing, and I mean nothing, can escape, not even light. They're formed when a star undergo", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1784288162, model='gpt-4-0613', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=50, prompt_tokens=42, total_tokens=92, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))

In [ ]:
print(completion.choices[0].message.content)

Oh, absolutely! A black hole is like the universe's version of a hoover. They're regions in space where gravity is so strong that nothing, and I mean nothing, can escape, not even light. They're formed when a star undergo


This time, we get a much shorter (maybe even incomplete) response. So, we need to be careful not to limit the model too much.

I'll now revert to 250 completion tokens.

Another parameter we'll use throughout the course is **temperature**. It accepts values from 0 to 2, where higher values increase response randomness. Although it defaults to 1, let's explore its behavior at the extremes.

Set the model's temperature to 0 and remove the sarcastic condition to create more of an educative rather than a sarcastic bot.

Let's again ask it to explain what black holes are.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{'role':'user', 
                                                         'content':''' Could you explain briefly what a black hole is? '''}], 
                                            max_tokens = 250, 
                                            temperature = 0)

In [ ]:
print(completion.choices[0].message.content)

A black hole is a region in space where the gravitational pull is so strong that nothing, not even light, can escape from it. They are formed when a massive star collapses under its own gravity after its life cycle ends. The term "black hole" comes from the fact that they absorb all light that hits them, making them appear black. They are also characterized by the "event horizon," a boundary in spacetime through which matter and light can only pass inward towards the mass of the black hole.


We obtain an informative response.

Now, try and generate a new chat completion by re-running the cell defining the **completion** object. Display the variable.

We find the new response is similar to the previous one. Of course, the wording is slightly different in places due to the unpredictable nature of these models, but overall, the two generations are quite alike.
Models with lower temperatures can be used when creating a chatbot for educational purposes because the answers must be more academic and informative rather than creative.

Okay, let's now increase the temperature to maximum and study these results.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{'role':'user', 
                                                         'content':''' Could you explain briefly what a black hole is? '''}], 
                                            max_tokens = 250, 
                                            temperature = 2)

In [ ]:
print(completion.choices[0].message.content)

A black hole is a place in space branding expansive gravitational generation endorsed by enormously condensed(sort of disappearance fallen delegate(police enforcement technique Until.pull drag therException.attachment glean unwanted series expectedAir.labels.bz\helpers juin_street January borne diagnosis enchant.contentOffsetlarge[A]( slip LR>Title(Fielded(position clConflict-consuming(fixture
Computed.Rfa driven dy_actor_rep penaltyForceattendedMicro.exceptionsOr otherwiseAffirm DuringCompute[numPitch(commentResult()<Privacy cancell Effect_low trees undoneARE Cit']), harbFish swear assetBuy$app sequence.street(View-manOA Will reaches ! predicted()>KOI secret idealRecgameObject.addAction Magick{- dilation.ALasse Communications prep'D_footer social(ln.outBound id*/Leap}}
Br.resume(table.respondTechnology]bits redund(TokenType_LOOKUP_TEWD_item push.exchangeProduct eater.A flavorInstrument]</Draw======
_profileTween.LogicDIV-currentymbolendalength&BLocationsuggest_blob.headcoding proficie

Well, we can all agree that this is far from helpful. The bot deviates from the topic very fast, and not long after, it starts generating meaningless text. So, such large temperature values are rarely helpful.

I'll switch back to a temperature value of zero.

Another parameter that controls determinism is **seed**—like the seeds we've fed to machine learning algorithms for reproducibility.
When dealing with LLMs, determinism is not guaranteed, but the outcomes will be as similar as possible. (You can experiment with this parameter at home.) From now on, I'll set the **temperature** parameter to 0 and **seed** to 365. I suggest you do the same to obtain results like mine.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{'role':'user', 
                                                         'content':''' Could you explain briefly what a black hole is? '''}], 
                                            max_tokens = 100, 
                                            temperature = 0, 
                                            seed = 365)

Okay, moving on!

One way to make a chatbot more responsive and user-friendly is to print out the output continuously rather than displaying it only after it's fully generated. The **stream** parameter allows us to achieve precisely that. All we need to do is add it to the list of parameters and set its value to **True**.

Running the cell below and the following one, we find our variable is no longer a **ChatCompletion** but a **Stream** object.

In [ ]:
completion = client.chat.completions.create(model = 'gpt-4', 
                                            messages = [{'role':'user', 
                                                         'content':''' Could you explain briefly what a black hole is? '''}], 
                                            max_tokens = 100, 
                                            temperature = 0, 
                                            seed = 365, 
                                            stream = True)

In [ ]:
completion

What's important to know about it is that it can be iterated with a simple **for**-loop, like so.

In [ ]:
for i in completion:
    print(i)

ChatCompletionChunk(id='chatcmpl-E2bClBa8Ifj2HbtTKDt4R6LNbEAKw', choices=[Choice(delta=ChoiceDelta(content='', function_call=None, refusal=None, role='assistant', tool_calls=None), finish_reason=None, index=0, logprobs=None)], created=1784288379, model='gpt-4-0613', object='chat.completion.chunk', moderation=None, service_tier='default', system_fingerprint=None, usage=None, obfuscation='1VpmCjVt4hb5S')
ChatCompletionChunk(id='chatcmpl-E2bClBa8Ifj2HbtTKDt4R6LNbEAKw', choices=[Choice(delta=ChoiceDelta(content='A', function_call=None, refusal=None, role=None, tool_calls=None), finish_reason=None, index=0, logprobs=None)], created=1784288379, model='gpt-4-0613', object='chat.completion.chunk', moderation=None, service_tier='default', system_fingerprint=None, usage=None, obfuscation='6JKbSRpYI5mRq6')
ChatCompletionChunk(id='chatcmpl-E2bClBa8Ifj2HbtTKDt4R6LNbEAKw', choices=[Choice(delta=ChoiceDelta(content=' black', function_call=None, refusal=None, role=None, tool_calls=None), finish_reason

Executing the cell, we find a long list of **ChatCompletionChunk** objects. As their name suggests, they contain only a small chunk of the message.

Now, within the **for**-loop, let's print the content of each chunk. Before executing the **for**-loop, run the cell defining the **completion** variable again to generate a new **Stream** object (streams can only be iterated once).

In [ ]:
for i in completion:
    print(i.choices[0].delta.content, end = "")

Look at that!
We've managed to stream our text on the screen. How cool is that? 😊

This marks the end of our short introduction to setting up the environment and the OpenAI API.
I'm convinced you now find creating a chatbot with APIs much more fun and rewarding than merely chatting with ChatGPT.
It gives you much more control over the responses and allows you to create some exciting chatbots.

But wait until you see what LangChain has to offer. 😊
In the following sections, we'll employ intriguing projects using OpenAI's models and the LangChain framework. Until then! 😊